**requerimientos**

In [2]:
!pip install pandas 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 1.8 MB/s eta 0:00:0000:0100:010m


In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# CONFIG
DATA_PATH =  "/Users/morenx/grind/gaynor/data/heart.csv"
sns.set_theme(style="whitegrid")                 # estilo visual 

# COMO KAGGLE YA ME DICE LA ESTRUCTURA DEL DF, APROVECHO PARA SEPARAR LOS TIPOS DE DATOS
NUMERIC_COLS = ["age", "trestbps", "chol", "thalach", "oldpeak"]
CATEGORICAL_COLS = ["cp", "sex", "fbs", "restecg", "exang", "slope", "ca", "thal"]


# Carga robusta del dataset 
def load_data(path: str) -> pd.DataFrame:
    """Carga el CSV y normaliza nombre de la columna objetivo a 'target'."""
    df = pd.read_csv(path)
    # Normaliza nombres de columnas (minusculas, sin espacios ni BOM)
    df.columns = [c.strip().lower().replace("\ufeff", "") for c in df.columns]

    # El objetivo aparece con distintos nombres segun la fuente del dataset
    if "target" not in df.columns:
        if "num" in df.columns:               # version UCI cruda: num en 0..4
            df["target"] = (df["num"] > 0).astype(int)
            df = df.drop(columns=["num"])
        elif "condition" in df.columns:        # version 'cleaned' de UCI
            df = df.rename(columns={"condition": "target"})
        elif "output" in df.columns:           # variante con columna 'output'
            df = df.rename(columns={"output": "target"})
        else:
            raise ValueError("No se encontro una columna objetivo reconocible.")

    # Algunas versiones UCI codifican faltantes como '?' -> convertir a NaN
    df = df.replace("?", np.nan)
    # Forzar tipo numerico en columnas que pudieron quedar como texto
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def main() -> None:
    # loaddata
    df = load_data(DATA_PATH)

    print("REVISION INICIAL DEL DATASET")
    print("\nPrimeras 5 filas (.head()):")
    print(df.head())
    print(f"\nDimensiones (.shape): {df.shape}  -> {df.shape[0]} filas, {df.shape[1]} columnas")

    print("\nTipos de datos (.dtypes):")
    print(df.dtypes)
    print("\nInformacion general (.info()):")
    df.info()

    # null
    nulls = df.isnull().sum()
    print("\nValores nulos por columna (.isnull().sum()):")
    print(nulls)

    # imputacion para la mediana para estabilidad ante outliers
    if nulls.sum() > 0:
        print(f"\nHay {int(nulls.sum())} valores faltantes. "
              "Imputando con  MEDIANA ")
        df = df.fillna(df.median(numeric_only=True))
    else:
        print("\n-> No se detectaron valores faltantes.")

    dups = int(df.duplicated().sum())
    if dups > 0:
        print(f"\n-> Se detectaron {dups} filas duplicadas. Se eliminan para evitar "
              "fuga de datos entre train/test.")
        df = df.drop_duplicates().reset_index(drop=True)
        print(f"   Dimensiones tras eliminar duplicados: {df.shape}")

    # stats decriptivas
    print("\nEstadisticas descriptivas:")
    print(df.describe())

    # Mantener solo columnas presentes (por si la variante difiere)
    num_cols = [c for c in NUMERIC_COLS if c in df.columns]
    cat_cols = [c for c in CATEGORICAL_COLS if c in df.columns]

    # EDA sobre variable objetivo

    counts = df["target"].value_counts().sort_index()
    pct = df["target"].value_counts(normalize=True).sort_index() * 100
    print("DISTRIBUCION DE LA VARIABLE OBJETIVO (target)")
    for cls in counts.index:
        print(f"  Clase {int(cls)}: {counts[cls]} muestras ({pct[cls]:.1f}%)")
    balance = "BALANCEADO" if pct.min() >= 40 else "DESBALANCEADO"
    print(f"  -> El dataset esta {balance} (clase minoritaria = {pct.min():.1f}%).")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.countplot(x="target", data=df, ax=axes[0], hue="target", palette="Set2", legend=False)
    axes[0].set_title("Distribucion de la clase objetivo")
    axes[0].set_xlabel("target (0 = sano, 1 = enfermedad)")
    axes[0].set_ylabel("Conteo")
    axes[1].pie(counts, labels=[f"Clase {int(c)}" for c in counts.index],
                autopct="%1.1f%%", colors=sns.color_palette("Set2"), startangle=90)
    axes[1].set_title("Proporcion de clases")
    fig.tight_layout()
    plt.close(fig)

    # ------------------------------------------------------------------
    # 3.3 Graficas obligatorias
    # ------------------------------------------------------------------
    # (1) Histogramas de variables numericas
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax, col in zip(axes.ravel(), num_cols):
        sns.histplot(df[col], kde=True, ax=ax, color="steelblue")
        ax.set_title(f"Histograma de {col}")
    for ax in axes.ravel()[len(num_cols):]:   # apagar ejes sobrantes
        ax.axis("off")
    fig.suptitle("Distribucion de variables numericas", fontsize=14)
    fig.tight_layout()
    plt.close(fig)

    # (2) Countplots de variables categoricas
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    for ax, col in zip(axes.ravel(), cat_cols):
        sns.countplot(x=col, data=df, ax=ax, hue=col, palette="viridis", legend=False)
        ax.set_title(f"Countplot de {col}")
    for ax in axes.ravel()[len(cat_cols):]:
        ax.axis("off")
    fig.suptitle("Frecuencia de variables categoricas", fontsize=14)
    fig.tight_layout()
    plt.close(fig)

    # (3) Boxplots por clase objetivo
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    for ax, col in zip(axes.ravel(), num_cols):
        sns.boxplot(x="target", y=col, data=df, ax=ax, hue="target",
                    palette="Set3", legend=False)
        ax.set_title(f"{col} por clase")
        ax.set_xlabel("target")
    for ax in axes.ravel()[len(num_cols):]:
        ax.axis("off")
    fig.suptitle("Variables numericas segun clase objetivo", fontsize=14)
    fig.tight_layout()
    plt.close(fig)

    # (4) Mapa de correlacion (heatmap)
    corr = df.corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
                square=True, ax=ax, cbar_kws={"shrink": 0.8})
    ax.set_title("Mapa de correlacion de Pearson")
    fig.tight_layout()
    plt.close(fig)

    # (5) Pairplot de variables seleccionadas (coloreado por clase)
    pair_cols = num_cols[:4] + ["target"]        # 4 numericas + objetivo
    pair = sns.pairplot(df[pair_cols], hue="target", palette="Set1",
                        diag_kind="kde", corner=True)
    pair.figure.suptitle("Pairplot de variables seleccionadas", y=1.02)
    plt.close(pair.figure)

    # (6) Barplot de correlacion con target (ranking)
    target_corr = corr["target"].drop("target").sort_values(ascending=False)
    print("CORRELACION DE CADA VARIABLE CON target (ordenada)")
    print(target_corr)

    fig, ax = plt.subplots(figsize=(10, 7))
    sns.barplot(x=target_corr.values, y=target_corr.index, ax=ax,
                hue=target_corr.index, palette="vlag", legend=False)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title("Correlacion de cada variable con target")
    ax.set_xlabel("Coeficiente de correlacion de Pearson")
    fig.tight_layout()
    plt.close(fig)

if __name__ == "__main__":
    main()

ValueError: No se encontro una columna objetivo reconocible.